In [1]:

import ultralytics
import roboflow
import cv2
import torch
import os
import yaml


In [ ]:


from roboflow import Roboflow
rf = Roboflow(api_key="zDiJbZBMUgWUCSOo6Fvx")
project = rf.workspace("prince-suman-3ohwu").project("hairnet-detection-vktcj")
version = project.version(2)
dataset = version.download("yolov8")


In [ ]:


from roboflow import Roboflow
rf = Roboflow(api_key="zDiJbZBMUgWUCSOo6Fvx")
project = rf.workspace("artyom").project("gloves-4nvvi")
version = project.version(1)
dataset = version.download("yolov8")


In [6]:
import os
import yaml

# Fix hairnet yaml
hairnet_path = os.path.abspath("Hairnet-Detection-2")
hairnet_yaml = {
    'names': ['No-Hairnet', 'hairnet'],
    'nc': 2,
    'train': os.path.join(hairnet_path, 'train', 'images'),
    'val': os.path.join(hairnet_path, 'valid', 'images'),
    'test': os.path.join(hairnet_path, 'test', 'images')
}
with open("Hairnet-Detection-2/data.yaml", "w") as f:
    yaml.dump(hairnet_yaml, f)

# Fix gloves yaml
gloves_path = os.path.abspath("gloves-1")
gloves_yaml = {
    'names': ['glove'],
    'nc': 1,
    'train': os.path.join(gloves_path, 'train', 'images'),
    'val': os.path.join(gloves_path, 'valid', 'images'),
    'test': os.path.join(gloves_path, 'test', 'images')
}
with open("gloves-1/data.yaml", "w") as f:
    yaml.dump(gloves_yaml, f)

print("Hairnet dataset path:", hairnet_path)
print("Gloves dataset path:", gloves_path)
print("YAML files fixed!")

Hairnet dataset path: C:\Users\ziada\PycharmProjects\CVproject\Hairnet-Detection-2
Gloves dataset path: C:\Users\ziada\PycharmProjects\CVproject\gloves-1
YAML files fixed!


In [8]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
hairnet_model = YOLO('yolov8n.pt')

# Train on hairnet dataset
hairnet_results = hairnet_model.train(
    data=os.path.abspath("Hairnet-Detection-2/data.yaml"),
    epochs=50,
    imgsz=640,
    batch=8,
    name='hairnet_model',
    device=0,  # GPU
    patience=15,
    save=True,
    plots=True
)

print("Hairnet training complete!")
print("Best model saved at: runs/detect/hairnet_model/weights/best.pt")

Ultralytics 8.4.33  Python-3.14.0 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ziada\PycharmProjects\CVproject\Hairnet-Detection-2\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=hairnet_model, nbs=64, nms=False, opset=None, optimize

In [2]:
from ultralytics import YOLO

gloves_model = YOLO('yolov8n.pt')


gloves_results = gloves_model.train(
    data=os.path.abspath("gloves-1/data.yaml"),
    epochs=50,
    imgsz=640,
    batch=8,
    name='gloves_model',
    device=0,
    patience=15,
    save=True,
    plots=True
)

print("Gloves training complete!")
print("Best model saved at: runs/detect/gloves_model/weights/best.pt")

Ultralytics 8.4.33  Python-3.14.0 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ziada\PycharmProjects\CVproject\gloves-1\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gloves_model, nbs=64, nms=False, opset=None, optimize=False, opti

In [1]:
from ultralytics import YOLO
import cv2
import matplotlib
matplotlib.use('Agg')  # prevents window popup, saves to file instead
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# Load both trained models
hairnet_model = YOLO('runs/detect/hairnet_model/weights/best.pt')
gloves_model = YOLO('runs/detect/gloves_model/weights/best.pt')

def check_cook(image_path):
    hairnet_results = hairnet_model(image_path, conf=0.5)
    gloves_results = gloves_model(image_path, conf=0.5)

    wearing_hairnet = False
    for result in hairnet_results:
        for box in result.boxes:
            if int(box.cls) == 1:
                wearing_hairnet = True


        glove_count = 0
        for result in gloves_results:
            for box in result.boxes:
                if int(box.cls) == 0:
                    glove_count += 1

        wearing_gloves = glove_count >= 2
    result = {
        "hairnet": wearing_hairnet,
        "gloves": wearing_gloves,
        "approved": wearing_hairnet and wearing_gloves,
        "message": ""
    }

    if wearing_hairnet and wearing_gloves:
        result["message"] = ";) Cook is wearing both hairnet and gloves. APPROVED."
    elif wearing_hairnet and not wearing_gloves:
        result["message"] = "WARNING Cook is wearing hairnet but NO gloves or only ONE glovee."
    elif not wearing_hairnet and wearing_gloves:
        result["message"] = "WARNING Cook is wearing gloves but NO hairnet."
    else:
        result["message"] = "WARNING Cook is wearing NEITHER hairnet nor gloves."

    print(result["message"])
    print("Hairnet:", wearing_hairnet)
    print("Gloves:", wearing_gloves)
    print("Approved:", result["approved"])

    # Save result image to file instead of showing window
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    hairnet_img = hairnet_results[0].plot()
    gloves_img = gloves_results[0].plot()
    axes[0].imshow(cv2.cvtColor(hairnet_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Hairnet Detection")
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(gloves_img, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Gloves Detection")
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('detection_result.png')
    plt.close()
    print("Result image saved to: detection_result.png")

    return result

root = tk.Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select a photo of the cook",
    filetypes=[("Image files", "*.jpg *.jpeg *.png")]
)

if file_path:
    print(f"Testing with: {file_path}")
    check_cook(file_path)
else:
    print("No file selected.")

Testing with: C:/Users/ziada/Desktop/sewar lel project/im55ages.jpeg

image 1/1 C:\Users\ziada\Desktop\sewar lel project\im55ages.jpeg: 512x640 1 No-Hairnet, 55.1ms
Speed: 4.8ms preprocess, 55.1ms inference, 22.4ms postprocess per image at shape (1, 3, 512, 640)

image 1/1 C:\Users\ziada\Desktop\sewar lel project\im55ages.jpeg: 512x640 2 gloves, 12.0ms
Speed: 2.2ms preprocess, 12.0ms inference, 2.6ms postprocess per image at shape (1, 3, 512, 640)
WARNING Cook is wearing gloves but NO hairnet.
Hairnet: False
Gloves: True
Approved: False
Result image saved to: detection_result.png
